# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amey05081999/Flyrank-Internship-Amey-Naik/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Lane:** lifecycle / review-priority modeling.

**Target:** `is_declining_label = (trend_direction == "down")`. The Week-4 notebook already established that the label is derived from `trend_direction`, so `trend_direction`, `trend_pct`, and the last-30/previous-30 comparison fields are excluded from modeling.

**Method choice:** I compare Logistic Regression, a shallow Decision Tree, and Random Forest, then select the simplest model that gives a useful improvement over the Week-4 rule baseline. Random Forest is the main candidate because the lane contains mixed numeric/categorical signals and likely nonlinear interactions (for example, visibility with position or freshness). It also supports a straightforward held-out permutation-importance analysis.

I do **not** choose a model because it is more complex. The decision is based on held-out `Precision@50`, with ROC-AUC and average precision reported as supporting diagnostics.

In [1]:
import requests
from pathlib import Path

# URL of the raw CSV file on GitHub
csv_url = "https://raw.githubusercontent.com/amey05081999/Flyrank-Internship-Amey-Naik/main/data/raw/content_refresh_anonymized.csv"

# Define the local path where the file should be saved
# This assumes Colab's default working directory is /content
local_dir = Path("/content/data/raw")
local_dir.mkdir(parents=True, exist_ok=True) # Create the directory if it doesn't exist
local_file_path = local_dir / "content_refresh_anonymized.csv"

# Download the file
response = requests.get(csv_url)
if response.status_code == 200:
    with open(local_file_path, "wb") as f:
        f.write(response.content)
    print(f"File downloaded successfully to {local_file_path}")
else:
    print(f"Failed to download file. Status code: {response.status_code}")


File downloaded successfully to /content/data/raw/content_refresh_anonymized.csv


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from sklearn.inspection import permutation_importance

# Locate the starter dataset from either the repo root or a Colab-mounted copy.
ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "raw" / "content_refresh_anonymized.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Expected data/raw/content_refresh_anonymized.csv. "
        "Run this notebook from the repository that contains the starter dataset."
    )

df = pd.read_csv(DATA_PATH)

# The Week-4 label definition.
df["is_declining_label"] = df["trend_direction"].eq("down").astype(int)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
print(f"Declining rate: {df['is_declining_label'].mean():.3f}")
print(f"Clients: {df['client_id'].nunique():,}")


Rows: 30,000
Columns: 45
Declining rate: 0.542
Clients: 32


## 2. Split design

I use a **client-grouped holdout**: approximately 20% of clients are held out completely for testing. This follows the Week-4/reference pipeline's validation principle and prevents pages from the same pseudonymized client appearing in both train and test.

The split is fixed with `random_state=42`. No target-derived fields are used to construct the split. The Week-4 baseline is recomputed on the held-out rows only, using the same transparent rule, so the baseline and learned models are compared on exactly the same test population and `Precision@50` metric.

In [3]:
# Leakage guardrails from the Week-4 signal audit / reference pipeline.
TARGET = "is_declining_label"

LEAKAGE_FIELDS = {
    TARGET,
    "trend_direction",
    "trend_pct",
    "is_declining",
    "health_score",
    "is_quick_win",
    "needs_ctr_fix",
    "needs_engagement_fix",
    "ai_opportunity",
    "is_underperformer",
    "is_initial_refresh_candidate",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
}

ID_FIELDS = {"content_id", "client_id"}

# Safe observable features available before the decline label.
CANDIDATE_FEATURES = [
    "search_volume", "competition", "competition_level", "cpc",
    "content_type", "main_intent", "word_count", "char_count",
    "provider_used", "model_used",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "content_age_days", "age_tier", "age_tier_order",
    "days_since_last_update", "freshness_tier", "word_count_tier",
    "char_count_tier", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "impression_tier", "position_tier",
]

FEATURES = [c for c in CANDIDATE_FEATURES if c in df.columns]
assert not set(FEATURES) & LEAKAGE_FIELDS
assert not set(FEATURES) & ID_FIELDS

X = df[FEATURES].copy()
y = df[TARGET].copy()
groups = df["client_id"].copy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

assert train_clients.isdisjoint(test_clients)

print(f"Train rows: {len(X_train):,} | Test rows: {len(X_test):,}")
print(f"Train clients: {len(train_clients):,} | Test clients: {len(test_clients):,}")
print(f"Train decline rate: {y_train.mean():.3f} | Test decline rate: {y_test.mean():.3f}")



Train rows: 23,837 | Test rows: 6,163
Train clients: 25 | Test clients: 7
Train decline rate: 0.550 | Test decline rate: 0.511


## 3. Train + compare vs my baseline

The comparison metric is **Precision@50**: among the 50 pages ranked highest for review, how many are actually in the observed declining class.

This is the same decision-oriented metric used by the Week-4/reference workflow. ROC-AUC and average precision are included only as supporting diagnostics.

The Week-4 rule is reproduced exactly from its notebook:

- visibility = percentile rank of `log1p(impressions_90d)`
- +1 for stale-and-visible (`days_since_last_update >= 180` and `impressions_90d >= 500`)
- +1 for the CTR opportunity (`impressions_90d >= 500`, position 1–20, CTR < 0.5%)

For fairness, that rule is evaluated **only on the held-out test rows**.

In [4]:
def precision_at_k(y_true, score, k=50):
    y_true = np.asarray(y_true)
    score = np.asarray(score)
    k = min(k, len(y_true))
    if k == 0:
        return np.nan
    order = np.argsort(-score, kind="mergesort")[:k]
    return float(y_true[order].mean())

def baseline_scores(frame):
    q = frame.copy()
    log_visibility = np.log1p(q["impressions_90d"].clip(lower=0))
    visibility_score = log_visibility.rank(method="average", pct=True)

    stale_visible = (
        (q["days_since_last_update"] >= 180)
        & (q["impressions_90d"] >= 500)
    )
    ctr_opportunity = (
        (q["impressions_90d"] >= 500)
        & (q["avg_position"] > 0)
        & (q["avg_position"] <= 20)
        & (q["ctr"] < 0.5)
    )
    return (
        visibility_score
        + stale_visible.astype(int)
        + ctr_opportunity.astype(int)
    ).to_numpy()

baseline_score = baseline_scores(df.iloc[test_idx])

numeric_features = [c for c in FEATURES if pd.api.types.is_numeric_dtype(X_train[c])]
categorical_features = [c for c in FEATURES if c not in numeric_features]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=50,
        class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=12, min_samples_leaf=10,
        class_weight="balanced_subsample", n_jobs=-1, random_state=42
    ),
}

fitted = {}
rows = [{
    "model": "Week-4 rule baseline",
    "precision_at_50": precision_at_k(y_test, baseline_score, 50),
    "roc_auc": np.nan,
    "average_precision": np.nan,
}]

for name, estimator in models.items():
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", estimator),
    ])
    pipe.fit(X_train, y_train)
    prob = pipe.predict_proba(X_test)[:, 1]

    fitted[name] = pipe
    rows.append({
        "model": name,
        "precision_at_50": precision_at_k(y_test, prob, 50),
        "roc_auc": roc_auc_score(y_test, prob),
        "average_precision": average_precision_score(y_test, prob),
    })

comparison = pd.DataFrame(rows)
baseline_p50 = comparison.loc[
    comparison["model"].eq("Week-4 rule baseline"), "precision_at_50"
].iloc[0]
comparison["delta_vs_baseline"] = comparison["precision_at_50"] - baseline_p50

display(comparison.sort_values("precision_at_50", ascending=False).reset_index(drop=True))

best_model_name = (
    comparison.loc[comparison["model"].ne("Week-4 rule baseline")]
    .sort_values(["precision_at_50", "average_precision"], ascending=False)
    .iloc[0]["model"]
)
best_model = fitted[best_model_name]
best_prob = best_model.predict_proba(X_test)[:, 1]

print(f"Selected model: {best_model_name}")
print(
    f"Observed held-out Precision@50: "
    f"{comparison.loc[comparison['model'].eq(best_model_name), 'precision_at_50'].iloc[0]:.3f}"
)
print(
    f"Baseline Precision@50: {baseline_p50:.3f}; "
    f"delta: {comparison.loc[comparison['model'].eq(best_model_name), 'delta_vs_baseline'].iloc[0]:+.3f}"
)



,model,precision_at_50,roc_auc,average_precision,delta_vs_baseline
0,Logistic Regression,0.62,0.577100,0.568648,0.20
1,Random Forest,0.60,0.612779,0.594860,0.18
2,Decision Tree,0.52,0.610374,0.581593,0.10
3,Week-4 rule baseline,0.42,NaN,NaN,0.00


Selected model: Logistic Regression
Observed held-out Precision@50: 0.620
Baseline Precision@50: 0.420; delta: +0.200


## 4. Errors and interpretation

I focus on two questions:

1. **What does the selected model lean on?** Permutation importance is measured on the held-out test set using ROC-AUC, so the importance is not taken from training impurity alone.
2. **Where is it wrong?** I inspect the 50 highest-scored pages and summarize false positives / false negatives by observable groups such as content type, position tier, and freshness tier.

These are decision-support diagnostics, not causal explanations. A high-importance feature means the held-out model performance changes when that feature is permuted; it does not prove that changing the feature would cause a page to decline.


In [5]:
# Held-out permutation importance.
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=5,
    random_state=42,
    n_jobs=-1,
)

importance = (
    pd.DataFrame({
        "feature": X_test.columns,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

display(importance.head(12))

# Error table for the selected model.
error_frame = df.iloc[test_idx].copy()
error_frame["actual_declining"] = y_test.to_numpy()
error_frame["model_score"] = best_prob
error_frame["model_predicted_declining"] = (best_prob >= 0.5).astype(int)

top50 = error_frame.sort_values("model_score", ascending=False).head(50).copy()
top50["top50_error"] = np.where(
    top50["actual_declining"].eq(1), "correct_declining", "false_positive"
)

print("Top-50 review queue:")
display(
    top50[
        [
            "actual_declining", "model_score",
            "content_type", "position_tier", "freshness_tier",
            "days_since_last_update", "impressions_90d", "ctr", "avg_position"
        ]
    ].head(10)
)

print("Top-50 outcome counts:")
display(top50["top50_error"].value_counts().rename_axis("outcome").to_frame("n"))

# Classification errors at the model's default 0.5 threshold.
cm = confusion_matrix(
    y_test,
    error_frame["model_predicted_declining"],
    labels=[0, 1],
)
cm_table = pd.DataFrame(
    cm,
    index=["actual_not_declining", "actual_declining"],
    columns=["predicted_not_declining", "predicted_declining"],
)
display(cm_table)

# Compact group-level error summaries.
for col in ["content_type", "position_tier", "freshness_tier"]:
    if col in error_frame.columns:
        group_error = (
            error_frame.groupby(col, dropna=False)
            .agg(
                n=("actual_declining", "size"),
                decline_rate=("actual_declining", "mean"),
                mean_score=("model_score", "mean"),
            )
            .sort_values("n", ascending=False)
            .head(10)
            .reset_index()
        )
        print(f"Error context by {col}:")
        display(group_error)

# Explicit self-checks.
assert not set(FEATURES) & LEAKAGE_FIELDS
assert not set(FEATURES) & ID_FIELDS
assert train_clients.isdisjoint(test_clients)
assert len(comparison) == 4



,feature,importance_mean,importance_std
0,days_with_impressions,0.044022,0.005092
1,days_with_sessions,0.034522,0.001859
2,content_age_days,0.032775,0.003211
3,users_90d,0.019088,0.001307
4,avg_position,0.016933,0.001992
5,word_count,0.006753,0.001168
6,position_tier,0.006534,0.001268
7,freshness_tier,0.004586,0.000607
8,age_tier_order,0.004313,0.001056
9,scroll_rate,0.003686,0.002323


Top-50 review queue:


,actual_declining,model_score,content_type,position_tier,freshness_tier,days_since_last_update,impressions_90d,ctr,avg_position
1537,1,0.919220,keyword article,page_1,0-30,20,128,0.00,4.2
23346,1,0.914569,keyword article,striking,0-30,8,138,0.00,14.6
3195,1,0.907289,keyword article,page_1,0-30,20,157,0.00,6.1
3626,1,0.898649,keyword article,striking,0-30,20,556,0.00,14.3
20736,0,0.895976,keyword article,striking,91-180,104,3115,0.00,12.8
10175,0,0.892189,keyword article,page_3_5,0-30,20,235,0.85,31.0
26956,1,0.887837,keyword article,striking,0-30,20,1398,0.07,15.9
27178,1,0.882405,keyword article,page_1,0-30,20,140079,0.01,7.6
11887,0,0.879435,keyword article,striking,91-180,102,289,0.69,18.8
18531,0,0.878774,keyword article,striking,91-180,104,166,0.60,16.1


Top-50 outcome counts:


,n
outcome,
correct_declining,31
false_positive,19


...

,content_type,n,decline_rate,mean_score
0,keyword article,6163,0.510952,0.489256


Error context by position_tier:


,position_tier,n,decline_rate,mean_score
0,page_1,2818,0.557133,0.522985
1,striking,1462,0.500000,0.529182
2,page_3_5,1273,0.480754,0.456077
3,top_3,330,0.412121,0.289294
4,deep,280,0.357143,0.327831


Error context by freshness_tier:


,freshness_tier,n,decline_rate,mean_score
0,0-30,4895,0.521757,0.467060
1,91-180,1218,0.466338,0.585203
2,31-90,50,0.540000,0.324950


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.